# Escherichia Coli exploration

**Author: T. Ruokokoski**

This notebook loads and inspects E. Coli core model.

- Display interactive map of reactions
- Run FBA to determine default maximum biomass production.
- Compare FBA and pFBA
- Display all exhange reactions
- Simulate anaerobic conditions by disabling oxygen uptake.
- Identify essential carbon sources by disabling them one at a time.
- Test alternative nitrogen sources
- Identify which base exchanges are essential for growth.
- Test which individual carbon sources (e.g. glucose, lactate, acetate) support growth.
- Inspect details of all reactions
- Inspect specific reactions and metabolites
- Run FVA on exchange reactions

In [1]:
import os
from cobra.io import load_model, read_sbml_model
from cobra import Model
from cobra.flux_analysis import pfba, flux_variability_analysis
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
warnings.filterwarnings("ignore", message="Solver status is 'infeasible'")

# Set paths
model_dir = "./models"

pd.set_option('display.max_rows', None)

### Visualizing Core Metabolism

This section loads the *E. coli core* metabolic model and visualizes it using an interactive Escher map. The map shows pathways and allows inspection of reaction fluxes.

In [2]:
from escher import Builder
from cobra.io import to_json

# Load model
model = load_model("textbook")

# Optionally: read from file
#model = read_sbml_model(os.path.join(model_dir, "e_coli_core.xml"))

# Load map as raw JSON string
with open("./models/e_coli_core.Core metabolism.json", "r") as f:
    map_json = f.read()

# Create Escher builder
b = Builder(
    model_json=to_json(model),  # JSON string of the model
    map_json=map_json,          # JSON string of the map
    reaction_scale_width=1.0
)

b


Builder()

### Basic Model Summary

Print statistics: number of reactions, metabolites, and genes for each model.

In [3]:
print(f"Model: {model.id}")
print(f"Reactions: {len(model.reactions)}") # biochemical transformations
print(f"Metabolites: {len(model.metabolites)}") # chemical compounds involved
print(f"Genes: {len(model.genes)}") # genes associated with enzymes that catalyze reactions
display(model.summary())

Model: e_coli_core
Reactions: 95
Metabolites: 72
Genes: 137


Metabolite,Reaction,Flux,C-Number,C-Flux
glc__D_e,EX_glc__D_e,10,6,100.00%
nh4_e,EX_nh4_e,4.765,0,0.00%
o2_e,EX_o2_e,21.8,0,0.00%
pi_e,EX_pi_e,3.215,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
co2_e,EX_co2_e,-22.81,1,100.00%
h2o_e,EX_h2o_e,-29.18,0,0.00%
h_e,EX_h_e,-17.53,0,0.00%


### Run Flux Balance Analysis (FBA)

Perform FBA to compute the maximum biomass production rate under default model constraints. This simulates optimal growth conditions and reveals the most active reactions in the metabolic network.

In [4]:
# Run Flux Balance Analysis (FBA)
default_solution = model.optimize()

# Default objective function is biomass production reaction
print(f"\nMax biomass growth rate: {default_solution.objective_value:.4f} mmol/gDW/hour")

# Get top 10 flux-carrying reactions
top_fluxes = default_solution.fluxes.sort_values(ascending=False).head(10)

top_reactions = []

for rxn_id, flux in top_fluxes.items():
    rxn = model.reactions.get_by_id(rxn_id)
    top_reactions.append({
        "Reaction ID": rxn.id,
        "Name": rxn.name,
        "Equation": rxn.reaction,
        "Flux": round(flux, 4)
    })

df_top = pd.DataFrame(top_reactions)
print("\nTop flux-carrying reactions:")
display(df_top)



Max biomass growth rate: 0.8739 mmol/gDW/hour

Top flux-carrying reactions:


,Reaction ID,Name,Equation,Flux
0,ATPS4r,ATP synthase (four protons for one ATP),adp_c + 4.0 h_e + pi_c <=> atp_c + h2o_c + 3.0...,45.5140
1,CYTBD,cytochrome oxidase bd (ubiquinol-8: 2 protons),2.0 h_c + 0.5 o2_c + q8h2_c --> h2o_c + 2.0 h_...,43.5990
2,NADH16,NADH dehydrogenase (ubiquinone-8 & 3 protons),4.0 h_c + nadh_c + q8_c --> 3.0 h_e + nad_c + ...,38.5346
3,EX_h2o_e,H2O exchange,h2o_e <=>,29.1758
4,EX_co2_e,CO2 exchange,co2_e <=>,22.8098
5,O2t,R o2 - transport-diffusion,o2_e <=> o2_c,21.7995
6,EX_h_e,H+ exchange,h_e <=>,17.5309
7,GAPD,glyceraldehyde-3-phosphate dehydrogenase,g3p_c + nad_c + pi_c <=> 13dpg_c + h_c + nadh_c,16.0235
8,ENO,enolase,2pg_c <=> h2o_c + pep_c,14.7161
9,GLCpts,D-glucose transport via PEP:Pyr PTS,glc__D_e + pep_c --> g6p_c + pyr_c,10.0000


### Parsimonius Flux Balance Analysis (pFBA)

pFBA finds a flux distribution which gives the optimal growth rate, but minimizes the total sum of flux. Both pFBA and FBA should return identical results within solver tolerances for the objective being optimized.

In [5]:
pfba_solution = pfba(model)

diff = abs(default_solution.fluxes["Biomass_Ecoli_core"] - pfba_solution.fluxes["Biomass_Ecoli_core"])
print(f"Difference in between standard FBA and pFBA: {diff}")

Difference in between standard FBA and pFBA: 3.6637359812630166e-15


### Exchange Reactions Overview

List and inspect all exchange reactions, which represent metabolite uptake and secretion between the cell and its environment. Useful for understanding how the model interfaces with external nutrients and byproducts.

In [6]:
print(f"Number of exchange reactions: {len(model.exchanges)}")
exchange_data = []

for rxn in model.reactions:
    if rxn.id.startswith('EX'):
        exchange_data.append({
            "Reaction ID": rxn.id,
            "Name": rxn.name,
            "Bounds": rxn.bounds,
            "Equation": rxn.reaction,
            "Flux": round(default_solution.fluxes[rxn.id], 4)
        })

df = pd.DataFrame(exchange_data)
display(df)


Number of exchange reactions: 20


,Reaction ID,Name,Bounds,Equation,Flux
0,EX_ac_e,Acetate exchange,"(0.0, 1000.0)",ac_e -->,0.0000
1,EX_acald_e,Acetaldehyde exchange,"(0.0, 1000.0)",acald_e -->,0.0000
2,EX_akg_e,2-Oxoglutarate exchange,"(0.0, 1000.0)",akg_e -->,0.0000
3,EX_co2_e,CO2 exchange,"(-1000.0, 1000.0)",co2_e <=>,22.8098
4,EX_etoh_e,Ethanol exchange,"(0.0, 1000.0)",etoh_e -->,0.0000
5,EX_for_e,Formate exchange,"(0.0, 1000.0)",for_e -->,0.0000
6,EX_fru_e,D-Fructose exchange,"(0.0, 1000.0)",fru_e -->,0.0000
7,EX_fum_e,Fumarate exchange,"(0.0, 1000.0)",fum_e -->,0.0000
8,EX_glc__D_e,D-Glucose exchange,"(-10.0, 1000.0)",glc__D_e <=>,-10.0000
9,EX_gln__L_e,L-Glutamine exchange,"(0.0, 1000.0)",gln__L_e -->,0.0000


In [7]:
model.medium

{'EX_co2_e': 1000.0,
 'EX_glc__D_e': 10.0,
 'EX_h_e': 1000.0,
 'EX_h2o_e': 1000.0,
 'EX_nh4_e': 1000.0,
 'EX_o2_e': 1000.0,
 'EX_pi_e': 1000.0}

### Anaerobic Growth Test

Simulate growth under anaerobic conditions by disabling oxygen uptake and re-running FBA. This tests whether the model can produce biomass without oxygen and reveals how flux distribution changes in its absence.

In [8]:
# Disable oxygen uptake
oxygen = model.reactions.get_by_id('EX_o2_e')
oxygen_lb_backup = oxygen.lower_bound
oxygen.lower_bound = 0.0

# Re-run FBA with no oxygen
anaerobic_solution = model.optimize()

print("\n=== Anaerobic growth test (oxygen uptake disabled) ===")
if anaerobic_solution.status == 'optimal':
    print(f"Biomass without oxygen: {anaerobic_solution.objective_value:.4f}")
    
    # Get top 10 flux-carrying reactions under anaerobic conditions
    top_fluxes_anaerobic = anaerobic_solution.fluxes.sort_values(ascending=False).head(10)

    anaerobic_reactions = []

    for rxn_id, flux in top_fluxes_anaerobic.items():
        rxn = model.reactions.get_by_id(rxn_id)
        anaerobic_reactions.append({
            "Reaction ID": rxn.id,
            "Name": rxn.name,
            "Equation": rxn.reaction,
            "Flux": round(flux, 4)
        })

    df_anaerobic = pd.DataFrame(anaerobic_reactions)
    print("\nTop flux-carrying reactions under anaerobic conditions:")
    display(df_anaerobic)

else:
    print("Optimization failed: no feasible solution without oxygen.")

# Restore original oxygen bound
oxygen.lower_bound = oxygen_lb_backup


=== Anaerobic growth test (oxygen uptake disabled) ===
Biomass without oxygen: 0.2117

Top flux-carrying reactions under anaerobic conditions:


,Reaction ID,Name,Equation,Flux
0,EX_h_e,H+ exchange,h_e <=>,30.5542
1,GAPD,glyceraldehyde-3-phosphate dehydrogenase,g3p_c + nad_c + pi_c <=> 13dpg_c + h_c + nadh_c,19.4373
2,ENO,enolase,2pg_c <=> h2o_c + pep_c,19.1207
3,FORti,formate transport via diffusion,for_c --> for_e,17.8047
4,PFL,pyruvate formate lyase,coa_c + pyr_c --> accoa_c + for_c,17.8047
5,EX_for_e,Formate exchange,for_e -->,17.8047
6,GLCpts,D-glucose transport via PEP:Pyr PTS,glc__D_e + pep_c --> g6p_c + pyr_c,10.0000
7,PGI,glucose-6-phosphate isomerase,g6p_c <=> f6p_c,9.9566
8,FBA,fructose-bisphosphate aldolase,fdp_c <=> dhap_c + g3p_c,9.7895
9,PFK,phosphofructokinase,atp_c + f6p_c --> adp_c + fdp_c + h_c,9.7895


### Identify Essential Carbon Sources

Test which carbon sources are strictly required for growth by individually disabling their uptake and re-running FBA. Sources whose removal eliminates or severely reduces biomass production are marked as essential.

Then test if organism can grow withoutany carbon source by disabling them all.

In [9]:
carbon_exchanges = [
    'EX_glc__D_e',   # D-Glucose
    'EX_fru_e',      # Fructose
    'EX_lac__D_e',   # D-Lactate
    'EX_pyr_e',      # Pyruvate
    'EX_ac_e',       # Acetate
    'EX_akg_e',      # 2-Oxoglutarate
    'EX_succ_e',     # Succinate
    'EX_fum_e',      # Fumarate
    'EX_mal__L_e',   # L-Malate
    'EX_etoh_e',     # Ethanol
    'EX_acald_e',    # Acetaldehyde
    'EX_for_e'       # Formate
]

# Enable all carbon source uptakes first
for ex in carbon_exchanges:
    if ex in model.reactions:
        model.reactions.get_by_id(ex).lower_bound = -10

# Test if any single carbon source is truly essential
non_redundant = []
threshold = 1e-9
for ex in carbon_exchanges:
    rxn = model.reactions.get_by_id(ex)
    original_lb = rxn.lower_bound
    rxn.lower_bound = 0

    sol = model.optimize()
    if sol.status != 'optimal' or sol.objective_value < threshold:
        non_redundant.append(ex)

    rxn.lower_bound = original_lb  # restore

print("\nCarbon sources that are truly essential:")
print(non_redundant)


Carbon sources that are truly essential:
[]


In [10]:
# Test if model can grow without any carbon sources

# Disable all carbon source uptake
for rxn_id in carbon_exchanges:
    rxn = model.reactions.get_by_id(rxn_id)
    rxn.lower_bound = 0.0

print(f"Current medium:\n{model.medium}\n")

threshold = 1e-9
sol = model.optimize()
failed = (sol.status != 'optimal')
very_low = (sol.objective_value is not None and sol.objective_value < threshold)

if failed or very_low:
    print("Carbon sources are essential: model cannot grow without them.")
else:
    print("Model can grow without carbon sources.")

Current medium:
{'EX_co2_e': 1000.0, 'EX_h_e': 1000.0, 'EX_h2o_e': 1000.0, 'EX_nh4_e': 1000.0, 'EX_o2_e': 1000.0, 'EX_pi_e': 1000.0}

Carbon sources are essential: model cannot grow without them.


In [11]:
# Allow only glucose uptake
for ex in carbon_exchanges:
    rxn = model.reactions.get_by_id(ex)
    if ex == 'EX_glc__D_e':
        rxn.lower_bound = -10
    else:
        rxn.lower_bound = 0.0

### Test Growth with Alternative Nitrogen Sources

This test evaluates whether E. coli can grow using different nitrogen sources (ammonia, glutamine, or glutamate) while keeping a fixed carbon source (glucose) and essential base exchanges enabled. Each nitrogen source is tested individually by setting its uptake and disabling all others.

In [12]:
# Save current medium and model state
original_medium = model.medium.copy()

carbon_source = 'EX_glc__D_e'

nitrogen_sources = [
    'EX_gln__L_e',   # L-Glutamine
    'EX_glu__L_e',   # L-Glutamate
    'EX_nh4_e'       # Ammonia
]
base_exchanges = [
    'EX_co2_e',
    'EX_h_e',
    'EX_h2o_e',
    'EX_o2_e',
    'EX_pi_e'
]

print("Testing nitrogen source alternatives with fixed carbon source:", carbon_source)
survival_results = {}

for nitrogen in nitrogen_sources:
    for rxn in model.exchanges:
        rxn.lower_bound = 0.0

    # Enable one carbon source
    model.reactions.get_by_id(carbon_source).lower_bound = -10

    # Enable base exchanges
    for ex in base_exchanges:
        model.reactions.get_by_id(ex).lower_bound = -1000

    # Enable only current nitrogen source
    model.reactions.get_by_id(nitrogen).lower_bound = -10

    solution = model.optimize()
    survived = solution.status == 'optimal'
    survival_results[nitrogen] = solution.objective_value if survived else 0.0

print("\nSurvival (growth) with each nitrogen source:")
for nit, growth in survival_results.items():
    status = f"YES (growth = {growth:.4f})" if growth > 0 else "NO"
    print(f"  {nit}: {status}")

# Restore original medium
model.medium = original_medium.copy()

Testing nitrogen source alternatives with fixed carbon source: EX_glc__D_e

Survival (growth) with each nitrogen source:
  EX_gln__L_e: YES (growth = 1.5034)
  EX_glu__L_e: YES (growth = 1.5401)
  EX_nh4_e: YES (growth = 0.8739)


### Identify Essential Base Exchange Reactions

This test evaluates which base exchange reactions (e.g. O₂, H⁺, H₂O, CO₂, phosphate) are essential for growth in the E. coli core model when glucose is provided as the sole carbon source and ammonia as the nitrogen source. Each base exchange is temporarily disabled to assess its impact on biomass production via FBA.


In [13]:
# Save current medium and model state
original_medium = model.medium.copy()

carbon_source = 'EX_glc__D_e'

nitrogen_source = 'EX_nh4_e'

base_exchanges = [
    'EX_co2_e',
    'EX_h_e',
    'EX_h2o_e',
    'EX_o2_e',
    'EX_pi_e'
]

for rxn in model.exchanges:
    rxn.lower_bound = 0.0

# Enable fixed carbon and nitrogen sources
model.reactions.get_by_id(carbon_source).lower_bound = -10
model.reactions.get_by_id(nitrogen_source).lower_bound = -10

for ex in base_exchanges:
    model.reactions.get_by_id(ex).lower_bound = -1000

# Test essentiality of each base exchange
essential_base = []
print("Testing essentiality of base exchanges:")

for ex in base_exchanges:
    rxn = model.reactions.get_by_id(ex)
    lb_orig = rxn.lower_bound
    rxn.lower_bound = 0.0

    sol = model.optimize()
    failed = (sol.status != 'optimal')
    very_low = (sol.objective_value is not None and sol.objective_value < 1e-3)

    if failed or very_low:
        essential_base.append(ex)
        status = "ESSENTIAL"
    else:
        status = "non-essential"

    print(f"  {ex}:\t{status}")

    rxn.lower_bound = lb_orig

print(f"\nTotal essential base reactions: {len(essential_base)}")

model.medium = original_medium.copy()

Testing essentiality of base exchanges:
  EX_co2_e:	non-essential
  EX_h_e:	non-essential
  EX_h2o_e:	non-essential
  EX_o2_e:	non-essential
  EX_pi_e:	ESSENTIAL

Total essential base reactions: 1


### Growth sensitivity to nutrient uptake

Tests how varying uptake of each non-carbon nutrient affects growth.  
Stores growth rates in `growth_profiles` and restores original bounds after testing.


In [14]:
uptake_exchanges = [
    rxn for rxn in model.exchanges
    if rxn.lower_bound < 0 and rxn.id not in carbon_exchanges
]

uptake_levels = np.linspace(0, 10, 5)
growth_profiles = {}

for rxn in uptake_exchanges:
    rxn_id = rxn.id
    original_lb = rxn.lower_bound
    growths = []

    for level in uptake_levels:
        rxn.lower_bound = -level
        sol = model.optimize()
        growth = sol.objective_value if sol.status == 'optimal' and sol.objective_value else 0.0
        growths.append(growth)

    growth_profiles[f"{rxn_id} ({rxn.name})"] = growths
    rxn.lower_bound = original_lb

df_growth = pd.DataFrame(growth_profiles, index=[f"{lvl:.1f}" for lvl in uptake_levels])
df_growth.index.name = "Uptake (mmol/gDW/h)"
display(df_growth.T.round(4))

Uptake (mmol/gDW/h),0.0,2.5,5.0,7.5,10.0
EX_co2_e (CO2 exchange),0.8739,0.8739,0.8739,0.8739,0.8739
EX_h_e (H+ exchange),0.8739,0.8739,0.8739,0.8739,0.8739
EX_h2o_e (H2O exchange),0.8739,0.8739,0.8739,0.8739,0.8739
EX_nh4_e (Ammonia exchange),0.0000,0.4585,0.8739,0.8739,0.8739
EX_o2_e (O2 exchange),0.2117,0.3017,0.3916,0.4778,0.5591
EX_pi_e (Phosphate exchange),-0.0000,0.6796,0.8739,0.8739,0.8739


### Growth on Different Carbon Sources

Each carbon source is tested individually by enabling only one carbon uptake at a time. Flux Balance Analysis is used to determine whether the carbon supports biomass production and to compare growth rates across different carbon sources.

In [15]:
print("\n=== Testing growth on different carbon sources ===")

results = []
for carbon in carbon_exchanges:

    for ex in carbon_exchanges:
        if ex in model.reactions:
            model.reactions.get_by_id(ex).lower_bound = 0

    model.reactions.get_by_id(carbon).lower_bound = -10

    sol = model.optimize()

    results.append({
        "Carbon source": carbon,
        "Uptake rate": round(sol.fluxes[carbon], 4),
        "Biomass Flux": round(sol.fluxes['Biomass_Ecoli_core'], 4)
    })

df_carbons = pd.DataFrame(results)
display(df_carbons)


=== Testing growth on different carbon sources ===


,Carbon source,Uptake rate,Biomass Flux
0,EX_glc__D_e,-10.0,0.8739
1,EX_fru_e,-10.0,0.8739
2,EX_lac__D_e,-10.0,0.3503
3,EX_pyr_e,-10.0,0.2912
4,EX_ac_e,-10.0,0.1733
5,EX_akg_e,-10.0,0.5288
6,EX_succ_e,-10.0,0.3976
7,EX_fum_e,-10.0,0.3707
8,EX_mal__L_e,-10.0,0.3707
9,EX_etoh_e,-10.0,0.3304


### Full Reaction Overview

All reactions in the model are collected into a DataFrame with basic metadata: reaction ID, name, equation, flux bounds, and associated genes. This provides a structured overview for further inspection or export.


In [16]:
reaction_data = []

for rxn in model.reactions:
    reaction_data.append({
        "Reaction ID": rxn.id,
        "Name": rxn.name,
        "Equation": rxn.reaction,
        "Bounds (mmol/gDW/h)": rxn.bounds,
        #"Subsystem": getattr(rxn, "subsystem", "N/A"),
        "Gene Associations": ", ".join(g.id for g in rxn.genes) if rxn.genes else "None"
    })

df_reactions = pd.DataFrame(reaction_data)
display(df_reactions)


,Reaction ID,Name,Equation,Bounds (mmol/gDW/h),Gene Associations
0,ACALD,acetaldehyde dehydrogenase (acetylating),acald_c + coa_c + nad_c <=> accoa_c + h_c + na...,"(-1000.0, 1000.0)","b0351, b1241"
1,ACALDt,R acetaldehyde reversible - transport,acald_e <=> acald_c,"(-1000.0, 1000.0)",s0001
2,ACKr,acetate kinase,ac_c + atp_c <=> actp_c + adp_c,"(-1000.0, 1000.0)","b3115, b1849, b2296"
3,ACONTa,"aconitase (half-reaction A, Citrate hydro-lyase)",cit_c <=> acon_C_c + h2o_c,"(-1000.0, 1000.0)","b0118, b1276"
4,ACONTb,"aconitase (half-reaction B, Isocitrate hydro-l...",acon_C_c + h2o_c <=> icit_c,"(-1000.0, 1000.0)","b0118, b1276"
5,ACt2r,R acetate reversible transport via proton - sy...,ac_e + h_e <=> ac_c + h_c,"(-1000.0, 1000.0)",None
6,ADK1,adenylate kinase,amp_c + atp_c <=> 2.0 adp_c,"(-1000.0, 1000.0)",b0474
7,AKGDH,2-Oxogluterate dehydrogenase,akg_c + coa_c + nad_c --> co2_c + nadh_c + suc...,"(0.0, 1000.0)","b0726, b0727, b0116"
8,AKGt2r,R 2 oxoglutarate reversible transport via - sy...,akg_e + h_e <=> akg_c + h_c,"(-1000.0, 1000.0)",b2587
9,ALCD2x,alcohol dehydrogenase (ethanol),etoh_c + nad_c <=> acald_c + h_c + nadh_c,"(-1000.0, 1000.0)","b1478, b1241, b0356"


### Inspect a Specific Reaction

Access a single reaction from the model by its index or ID.

In [17]:
#model.reactions[29]

model.reactions.get_by_id("FORti")

Reaction identifier,FORti
Name,formate transport via diffusion
Memory address,0x7416201e87d0
Stoichiometry,for_c --> for_e Formate --> Formate
GPR,b0904 or b2492
Lower bound,0.0
Upper bound,1000.0


### Inspect a Specific Metabolite

Retrieve a metabolite from the model using its ID (`"o2_e"` for extracellular oxygen) or index.

In [18]:
model.metabolites.get_by_id("o2_e")

#model.metabolites[3]

Metabolite identifier,o2_e
Name,O2
Memory address,0x741620174ec0
Formula,O2
Compartment,e
In 2 reaction(s),"EX_o2_e, O2t"


### Running Flux Variablity Analysis (FVA)

Runs FVA to determine the minimum and maximum flux values that each reaction can carry while maintaining optimal growth. Useful for assessing metabolic flexibility.

In [19]:


# Restore original medium
model.medium = original_medium.copy()

exchange_rxns = [rxn for rxn in model.exchanges]

# Run FVA on exchange reactions
flux_variability_analysis(model, exchange_rxns)

,minimum,maximum
EX_ac_e,0.000000,3.232258e-14
EX_acald_e,0.000000,-8.237973e-15
EX_akg_e,0.000000,1.212097e-14
EX_co2_e,22.809833,2.280983e+01
EX_etoh_e,0.000000,1.876795e-14
EX_for_e,0.000000,-6.711889e-14
EX_fru_e,0.000000,0.000000e+00
EX_fum_e,0.000000,0.000000e+00
EX_glc__D_e,-10.000000,-1.000000e+01
EX_gln__L_e,0.000000,-4.345365e-15


### Inspect training data

In [20]:
datafile = "./data/2025-07-28_full_training_data_98066_samples.csv" # log-uniform sampling
#datafile = "./data/2025-06-24_full_training_data_149990_samples.csv" # skewed power

df = pd.read_csv(datafile)

mean = df['ME1_flux'].mean()
std = df['ME1_flux'].std()

print(f"ME1_flux mean: {mean}")
print(f"ME1_flux standard deviation: {std}")

# Rows where ME1_flux is not 0 and not NaN
filtered_df = df[(df['ME1_flux'] != 0) & (~pd.isna(df['ME1_flux']))]

# Rows where ME1_flux is 0 or NaN
nonactive_df = df[(df['ME1_flux'] == 0) | (pd.isna(df['ME1_flux']))].sample(5, random_state=1)

# Columns to show: first 10 plus ME1_flux
columns_to_show = list(df.columns[:14])
if 'ME1_flux' not in columns_to_show:
    columns_to_show.append('ME1_flux')

print(f"Active ME1_flux rows (nonzero and not NaN): {len(filtered_df)}")
display(filtered_df[columns_to_show].head(10))

print(f"Non-active ME1_flux rows (0 or NaN)")
display(nonactive_df[columns_to_show])

ME1_flux mean: 0.27056239127818155
ME1_flux standard deviation: 1.6028592621497355
Active ME1_flux rows (nonzero and not NaN): 4117


,EX_glc__D_e,EX_fru_e,EX_lac__D_e,EX_pyr_e,EX_ac_e,EX_akg_e,EX_succ_e,EX_fum_e,EX_mal__L_e,EX_etoh_e,EX_acald_e,EX_for_e,EX_gln__L_e,EX_glu__L_e,ME1_flux
45,NaN,0.13,1.56,NaN,0.12,0.98,NaN,0.47,1.55,0.27,NaN,0.33,4.68,6.22,6.305885
46,0.15,1.21,NaN,0.78,0.23,7.95,NaN,0.19,NaN,NaN,NaN,6.83,NaN,43.77,10.235342
47,0.39,NaN,0.12,2.48,0.23,NaN,1.37,NaN,NaN,6.78,7.09,NaN,46.30,NaN,6.082412
53,1.39,NaN,6.50,NaN,NaN,NaN,1.22,NaN,1.08,NaN,NaN,2.11,NaN,36.96,7.362483
54,4.48,NaN,3.07,NaN,NaN,0.11,0.47,0.18,NaN,NaN,NaN,NaN,41.08,41.77,1.175026
55,NaN,NaN,0.73,0.23,0.61,0.11,NaN,NaN,NaN,4.79,NaN,NaN,37.93,NaN,10.014278
56,NaN,0.11,8.66,NaN,8.85,NaN,0.18,3.28,0.11,NaN,NaN,3.16,16.86,NaN,2.104365
74,NaN,NaN,0.32,2.91,NaN,NaN,5.04,4.16,NaN,2.20,0.40,NaN,21.77,8.34,3.223187
128,1.44,NaN,2.16,NaN,0.12,0.15,NaN,NaN,NaN,NaN,4.43,NaN,39.32,NaN,4.547171
171,3.64,NaN,0.42,NaN,0.91,NaN,NaN,NaN,NaN,NaN,NaN,0.40,9.54,NaN,4.304169


Non-active ME1_flux rows (0 or NaN)


,EX_glc__D_e,EX_fru_e,EX_lac__D_e,EX_pyr_e,EX_ac_e,EX_akg_e,EX_succ_e,EX_fum_e,EX_mal__L_e,EX_etoh_e,EX_acald_e,EX_for_e,EX_gln__L_e,EX_glu__L_e,ME1_flux
65531,4.74,NaN,0.51,0.23,NaN,NaN,NaN,8.14,1.26,4.70,6.39,0.75,15.56,33.16,0.0
86739,NaN,2.51,NaN,NaN,NaN,NaN,1.89,0.21,0.44,NaN,NaN,0.18,NaN,NaN,0.0
73492,NaN,NaN,NaN,NaN,0.26,2.20,7.38,NaN,NaN,NaN,NaN,4.19,NaN,NaN,0.0
44801,NaN,0.53,0.23,NaN,0.13,0.67,9.31,NaN,NaN,0.95,NaN,NaN,NaN,19.44,0.0
75169,NaN,NaN,0.37,0.13,8.02,0.13,NaN,9.94,2.63,NaN,NaN,0.26,NaN,27.77,0.0


In [21]:
[f"{rxn.id}_flux" for rxn in model.reactions]

['ACALD_flux',
 'ACALDt_flux',
 'ACKr_flux',
 'ACONTa_flux',
 'ACONTb_flux',
 'ACt2r_flux',
 'ADK1_flux',
 'AKGDH_flux',
 'AKGt2r_flux',
 'ALCD2x_flux',
 'ATPM_flux',
 'ATPS4r_flux',
 'Biomass_Ecoli_core_flux',
 'CO2t_flux',
 'CS_flux',
 'CYTBD_flux',
 'D_LACt2_flux',
 'ENO_flux',
 'ETOHt2r_flux',
 'EX_ac_e_flux',
 'EX_acald_e_flux',
 'EX_akg_e_flux',
 'EX_co2_e_flux',
 'EX_etoh_e_flux',
 'EX_for_e_flux',
 'EX_fru_e_flux',
 'EX_fum_e_flux',
 'EX_glc__D_e_flux',
 'EX_gln__L_e_flux',
 'EX_glu__L_e_flux',
 'EX_h_e_flux',
 'EX_h2o_e_flux',
 'EX_lac__D_e_flux',
 'EX_mal__L_e_flux',
 'EX_nh4_e_flux',
 'EX_o2_e_flux',
 'EX_pi_e_flux',
 'EX_pyr_e_flux',
 'EX_succ_e_flux',
 'FBA_flux',
 'FBP_flux',
 'FORt2_flux',
 'FORti_flux',
 'FRD7_flux',
 'FRUpts2_flux',
 'FUM_flux',
 'FUMt2_2_flux',
 'G6PDH2r_flux',
 'GAPD_flux',
 'GLCpts_flux',
 'GLNS_flux',
 'GLNabc_flux',
 'GLUDy_flux',
 'GLUN_flux',
 'GLUSy_flux',
 'GLUt2r_flux',
 'GND_flux',
 'H2Ot_flux',
 'ICDHyr_flux',
 'ICL_flux',
 'LDH_D_flux',
 '